In [1]:
import ee
import xarray
import rioxarray
import rasterio
from rasterio.transform import from_origin
import threading
import warnings
import dask.dataframe as dd
import httplib2
from dask.distributed import Client
import json
import os
import numpy as np

In [2]:
#http_transport = httplib2.Http(disable_ssl_certificate_validation=True)
#ee.Initialize(
#    opt_url='https://earthengine-highvolume.googleapis.com',
#    http_transport=http_transport, project='calfuels')
json_key = "test_key.json"
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

In [2]:
initialize_dict = {
    'opt_url': 'https://earthengine-highvolume.googleapis.com',
    'project': 'calfuels'
}

In [ ]:
# Lazy operation! Does not load the raster into memory!!
da = xarray.open_dataset("https://github.com/mapbox/rasterio/raw/1.2.1/tests/data/RGB.byte.tif")

In [ ]:
###############################
### TESTING MY PACKAGE CODE ###
###############################

In [5]:
import sys
import os
import ee
import os

ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com', project='calfuels')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from input_driver import EarthEngineReader

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

FAILING_CHUNKS = {
    'time': 48,
    'X': 512,
    'Y': 512
}
        
chunk_size = {
            'time': 3,
            'X': 2097,
            'Y': 2097
}
chunk_size  = {
            'time': 48,
            'X': 512,
            'Y': 256
}

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'chunks': chunk_size,
            'map_function': prep_sr_l8
        }
landsat_xarray = EarthEngineReader(parameters)

In [3]:
print(landsat_xarray.dataset)

<xarray.Dataset>
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 2020-12-29T18:57:32.281000 ... 2020-12-29T...
  * X        (X) float64 -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 -5.992e+05 -5.991e+05 -5.99e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_2_bands:  SR_B7,SR_B5,

In [7]:
import numpy as np
import pandas as pd
import xarray as xr

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    #'SR_B1': (['time', 'X', 'Y'], data),
    #'SR_B2': (['time', 'X', 'Y'], data),
    #'SR_B3': (['time', 'X', 'Y'], data),
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
    #'SR_B6': (['time', 'X', 'Y'], data),
    #'ST_EMSD': (['time', 'X', 'Y'], data),
    #'ST_QA': (['time', 'X', 'Y'], data),
    #'ST_TRAD': (['time', 'X', 'Y'], data),
    #'ST_URAD': (['time', 'X', 'Y'], data),
    #'QA_PIXEL': (['time', 'X', 'Y'], data),
    #'QA_RADSAT': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 512, 'Y': 256}

# Create the dataset
dataset = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)


<xarray.Dataset>
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 2020-12-29T18:57:32.281000 ... 2020-12-31T...
  * X        (X) float64 -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 -5.992e+05 -5.991e+05 -5.99e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
Attributes:
    date_range:            [1365638400000, 1654560000000]
    description:           <p>This dataset contains atmospherically corrected...
    keywords:              ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'la...
    period:                0
    visualization_2_max:   30000.0
    visualization_2_min:   0.0
    visualization_2_name:  Shortwave Infrared (753)
    crs:                   EPSG:3310


In [4]:
import dask.dataframe as dd
import xarray as xr
import numpy as np
import pandas as pd
# Convert to Dask DataFrame with matching partitions
def dataset_to_dask_dataframe(ds):
    # Convert each DataArray to a Dask DataFrame
    dfs = []
    for var in ds.data_vars:
        df = ds[var].to_dataframe().reset_index()
        dfs.append(df)
    
    # Concatenate DataFrames along columns
    ddf = dd.concat(dfs, axis=1)
    
    # Drop duplicate coordinate columns that resulted from the concatenation
    ddf = ddf.loc[:, ~ddf.columns.duplicated()]
    
    return ddf

dask_df = dataset_to_dask_dataframe(dataset)

In [5]:
dask_df = dask_df.repartition(npartitions=756)

In [6]:
print(dask_df)

Dask DataFrame Structure:
                           time        X        Y    SR_B4    SR_B5
npartitions=756                                                    
0                datetime64[ns]  float64  float64  float32  float32
381311                      ...      ...      ...      ...      ...
...                         ...      ...      ...      ...      ...
287890343                   ...      ...      ...      ...      ...
288271655                   ...      ...      ...      ...      ...
Dask Name: repartition-merge, 3 graph layers


In [8]:
import xarray as xr

# Assuming 'ds' is your xarray dataset
chunks = dataset.chunks

# Calculate the total number of chunks
total_chunks = 1
for dim, chunk_sizes in chunks.items():
    total_chunks *= len(chunk_sizes)

print(f"Total number of chunks: {total_chunks}")


Total number of chunks: 756


In [3]:
print(landsat_xarray.dataset)

<xarray.Dataset>
Dimensions:        (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time           (time) datetime64[ns] 2020-12-29T18:57:32.281000 ... 2020-...
  * X              (X) float64 -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y              (Y) float64 -5.992e+05 -5.991e+05 ... 4.584e+05 4.585e+05
Data variables: (12/19)
    SR_B1          (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B2          (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B3          (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B4          (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B5          (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B6          (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    ...             ...
    ST_EMSD        (time, X, Y) float32 dask.array<chunksize=(3, 512, 256), meta=

In [3]:
rechunked_dataset = dataset.chunk({'time': 3, 'X': 3000, 'Y': 3000})

In [1]:
import sys
import os

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from initialize_dask import DaskHandler

handler = DaskHandler()
handler.create_local_cluster(n_workers=8, threads_per_worker=1, memory_limit="32GB")
handler.initialize_earth_engine_dask_workers()
# If the crs is 4326 or anything geographic, I need a check to ensure that lon/lat are
# the dimension names!
# Maybe a method to close the cluster when done?

In [4]:
import dask.array as da
import xarray as xr
from dask.distributed import performance_report

def compute_ndvi(dataset):
    """
    Compute NDVI from an xarray Dataset using Dask.
    
    Parameters:
    dataset (xarray.Dataset): The input dataset containing NIR and Red bands.
    
    Returns:
    xarray.DataArray: NDVI values computed from the input dataset.
    """
    # Access the NIR (SR_B5) and Red (SR_B4) bands
    nir = dataset['SR_B5']
    red = dataset['SR_B4']
    
    # Ensure the arrays are dask arrays
    nir = nir.data if isinstance(nir, xr.DataArray) else nir
    red = red.data if isinstance(red, xr.DataArray) else red
    
    # Compute NDVI using Dask array operations
    ndvi = (nir - red) / (nir + red)
    
    # Convert the resulting dask array back to a DataArray
    ndvi_da = xr.DataArray(ndvi, dims=dataset['SR_B5'].dims, coords=dataset['SR_B5'].coords, name='NDVI')
    dataset['NDVI'] = ndvi_da
    
    return dataset

# Example usage with the dataset created earlier
ndvi = compute_ndvi(dataset)
with performance_report(filename='dask-report-nodf.html'):
    ndvi.compute()

In [4]:
dataset = landsat_xarray.dataset

# Select the first chunk (first time step, top-left corner) for SR_B5 and SR_B4
nir_chunk = dataset['SR_B5'].data.blocks[0, 0, 0]
red_chunk = dataset['SR_B4'].data.blocks[0, 0, 0]

# Define the NDVI computation function
def compute_ndvi(nir, red):
    return (nir - red) / (nir + red)

# Compute the NDVI for the selected chunk
ndvi_chunk = compute_ndvi(nir_chunk, red_chunk)

ndvi_chunk.compute()
# Visualize the task graph
#ndvi_chunk.visualize(filename='ndvi_chunk_task_graph_small.png')

In [5]:
dask_df = handler.dataset_to_dask_dataframe(landsat_xarray.dataset)
dask_df.dask.visualize(filename='high-level-graph.svg')

r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\IPython\core\interactiveshell.py:3550: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  exec(code_obj, self.user_global_ns, self.user_ns)
r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\IPython\core\interactiveshell.py:3550: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': Fa

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [9]:
import dask
with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    dask_df = handler.dataset_to_dask_dataframe(dataset)

In [19]:
dask_df = dataset.to_dask_dataframe(limit='128 MiB')#.repartition(partition_size='128 MiB')

TypeError: to_dask_dataframe() got an unexpected keyword argument 'limit'

In [13]:
print(dask_df)

Dask DataFrame Structure:
                          time        X        Y    SR_B4    SR_B5
npartitions=18                                                    
0               datetime64[ns]  float64  float64  float32  float32
16777216                   ...      ...      ...      ...      ...
...                        ...      ...      ...      ...      ...
285212672                  ...      ...      ...      ...      ...
288271655                  ...      ...      ...      ...      ...
Dask Name: concat-indexed, 1 graph layer


In [17]:
#dask_df = dask_df.repartition(npartitions=100)
def compute_ndvi(dask_df):
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    return (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])

    return dask_df

#single_partition = dask_df.partitions[0]
#result = single_partition.map_partitions(compute_ndvi)
dask_df['NDVI'] = dask_df.map_partitions(compute_ndvi)

In [18]:
dask_df['NDVI'].compute()

0            0.0
1            0.0
2            0.0
3            0.0
4            0.0
            ... 
288271651    0.0
288271652    0.0
288271653    0.0
288271654    0.0
288271655    0.0
Name: NDVI, Length: 288271656, dtype: float32

In [8]:
import dask
dask.visualize(dataset['NDVI'], filename="dask_xarray_task_graph.svg")

2024-07-15 14:23:39,334 - distributed.utils_perf - WARNING - full garbage collections took 15% CPU time recently (threshold: 10%)
2024-07-15 14:23:48,449 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)
2024-07-15 14:23:57,300 - distributed.utils_perf - WARNING - full garbage collections took 16% CPU time recently (threshold: 10%)
2024-07-15 14:24:06,173 - distributed.utils_perf - WARNING - full garbage collections took 17% CPU time recently (threshold: 10%)
2024-07-15 14:24:14,261 - distributed.utils_perf - WARNING - full garbage collections took 17% CPU time recently (threshold: 10%)
2024-07-15 14:24:23,498 - distributed.utils_perf - WARNING - full garbage collections took 17% CPU time recently (threshold: 10%)
2024-07-15 14:24:31,535 - distributed.utils_perf - WARNING - full garbage collections took 18% CPU time recently (threshold: 10%)
2024-07-15 14:24:40,303 - distributed.utils_perf - WARNING - full garbage collections took

In [4]:
dask_df = handler.dataset_to_dask_dataframe(landsat_xarray.dataset)

def compute_ndvi(dask_df):
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    dask_df['NDVI'] = (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])

    return dask_df

from udf_tuner import UserDefinedFunctionTuner
tuner = UserDefinedFunctionTuner(dask_df)
tuner.apply_function(compute_ndvi)

r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\statistics.py:168: RuntimeWarning: overflow encountered in long_scalars
  partials[d] = partials_get(d, 0) + n


In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from dask.distributed import Client, LocalCluster

client = Client(LocalCluster(n_workers=8, threads_per_worker=1, memory_limit='32GiB'))

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 512, 'Y': 256}

# Create the dataset
dataset = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)

dask_df = dataset.to_dask_dataframe().repartition(partition_size='128 MiB')

def compute_ndvi(dask_df):
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    return (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])

dask_df['NDVI'] = dask_df.map_partitions(compute_ndvi)
dask_df['NDVI'].compute()

r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\IPython\core\interactiveshell.py:3490: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  if await self.run_code(code, result, async_=asy):
r:\Users\adrianom.UNR\AppData\Local\anaconda3\envs\big-raster\lib\site-packages\IPython\core\interactiveshell.py:3490: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': Fa

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from dask.distributed import Client, LocalCluster
from dask.distributed import performance_report

client = Client(LocalCluster(n_workers=8, threads_per_worker=1, memory_limit='32GiB'))

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 512, 'Y': 256}

# Create the dataset
dataset = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)

def compute_ndvi(dataset):
    """
    Compute NDVI from an xarray Dataset using Dask.
    
    Parameters:
    dataset (xarray.Dataset): The input dataset containing NIR and Red bands.
    
    Returns:
    xarray.DataArray: NDVI values computed from the input dataset.
    """
    # Access the NIR (SR_B5) and Red (SR_B4) bands
    nir = dataset['SR_B5']
    red = dataset['SR_B4']
    
    # Ensure the arrays are dask arrays
    nir = nir.data if isinstance(nir, xr.DataArray) else nir
    red = red.data if isinstance(red, xr.DataArray) else red
    
    # Compute NDVI using Dask array operations
    ndvi = (nir - red) / (nir + red)
    
    # Convert the resulting dask array back to a DataArray
    ndvi_da = xr.DataArray(ndvi, dims=dataset['SR_B5'].dims, coords=dataset['SR_B5'].coords, name='NDVI')
    dataset['NDVI'] = ndvi_da
    
    return dataset

# Example usage with the dataset created earlier
ndvi = compute_ndvi(dataset)
with performance_report(filename='dask-report-nodf.html'):
    ndvi.compute()